# Checkpoint and Resume

Save and reload a trained model with `CheckpointManager`.

In [ ]:
import tempfile
from pathlib import Path

import torch
from boltzmann_generators.flows import FlowModel, GaussianPrior, RealNVP
from boltzmann_generators.services import CheckpointManager
from boltzmann_generators.training import TrainConfig, Trainer
from boltzmann_generators.energies import DoubleWell2D

torch.manual_seed(3)
device = "cpu"
energy = DoubleWell2D()
model = FlowModel(GaussianPrior(2), RealNVP(dim=2, num_layers=2, hidden_dim=16, mask="halves"))
Trainer(model, energy, TrainConfig(n_epochs=5, batch_size=32), device=device).fit()

x_probe = torch.randn(8, 2)
before = model.log_prob(x_probe).detach().clone()

mgr = CheckpointManager()
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "model.pt"
    mgr.save(model, path, config={"epochs": 5}, metrics={"loss": 1.0})
    model2 = FlowModel(GaussianPrior(2), RealNVP(dim=2, num_layers=2, hidden_dim=16, mask="halves"))
    model2, meta = mgr.load(path, model2)
    after = model2.log_prob(x_probe).detach()

print("Checkpoint metrics:", meta["metrics"])
print("Log-prob match:", torch.allclose(before, after).item())